# 🚕 Uber Fare Prediction Model — Complete Analysis

**Business objective:** predict `fare_amount` before the trip is completed using the instructor-provided 50K dataset.

**Dataset note:** `passenger_count` is absent, so passenger-count analysis is reported as **N/A** rather than fabricated. Coordinate-derived Haversine distance is calculated for validation, but the active model uses supplied `distance_km` because the coordinates are internally inconsistent with it.

In [ ]:
# Cell 1 — Imports and project paths
# Centralize imports and file paths so the notebook is reproducible.
from pathlib import Path
import json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = Path.cwd() if (Path.cwd() / 'train.py').exists() else Path.cwd().parent
RAW_PATH = BASE / 'dataset/uber_trips_dataset_50k.csv'
MODEL_PATH = BASE / 'models/uber_fare_model.pkl'
META_PATH = BASE / 'models/model_metadata.json'
CV_PATH = BASE / 'models/cross_validation_results.csv'
FINAL_PATH = BASE / 'models/final_test_metrics.csv'
print('Project root:', BASE.resolve())

## Phase 1 — Data Understanding
Inspect shape, columns, data types, missing values, duplicates, and descriptive statistics before cleaning.

In [ ]:
# Cell 2 — Load and inspect the raw dataset
# No modeling assumptions are made before checking the supplied schema.
raw = pd.read_csv(RAW_PATH)
print('Shape:', raw.shape)
print('Columns:', raw.columns.tolist())
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
print('Missing values:', int(raw.isna().sum().sum()))
print('Duplicate rows:', int(raw.duplicated().sum()))
print('Duplicate trip IDs:', int(raw['trip_id'].duplicated().sum()))
print('passenger_count present:', 'passenger_count' in raw.columns)
display(raw.describe(include='all').T)

## Phase 2 — Data Cleaning
Validate fares, distances, timestamps, coordinates, and duplicate records. Passenger-count validation is impossible because the column is not in the supplied data.

In [ ]:
# Cell 3 — Explicit cleaning checks
# Keep every rule visible so the data-quality decisions are auditable.
raw['pickup_time'] = pd.to_datetime(raw['pickup_time'], errors='coerce')
raw['drop_time'] = pd.to_datetime(raw['drop_time'], errors='coerce')
valid = (
    (raw['fare_amount'] > 0)
    & (raw['distance_km'] > 0)
    & (raw['drop_time'] > raw['pickup_time'])
    & raw['pickup_lat'].between(-90, 90)
    & raw['drop_lat'].between(-90, 90)
    & raw['pickup_lng'].between(-180, 180)
    & raw['drop_lng'].between(-180, 180)
)
clean = raw.loc[valid].drop_duplicates().copy()
completed = clean.loc[clean['status'].eq('Completed')].copy()
print('Valid cleaned rows:', len(clean))
print('Completed modeling rows:', len(completed))

## Phase 3 — Feature Engineering
Create pickup-time features and calculate Haversine distance as the assignment-requested coordinate-distance diagnostic.

In [ ]:
# Cell 4 — Create pickup date/time features
# These are available at trip start and therefore do not leak post-trip information.
pickup = completed['pickup_time']
completed['pickup_year'] = pickup.dt.year
completed['pickup_month'] = pickup.dt.month
completed['pickup_day'] = pickup.dt.day
completed['pickup_hour'] = pickup.dt.hour
completed['day_of_week'] = pickup.dt.dayofweek
completed['day_name'] = pickup.dt.day_name()
completed['is_weekend'] = pickup.dt.dayofweek.isin([5, 6]).astype(int)
completed['is_rush_hour'] = pickup.dt.hour.isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
display(completed[['pickup_time','pickup_year','pickup_month','pickup_day','pickup_hour','day_of_week']].head())

In [ ]:
# Cell 5 — Calculate and validate Haversine distance
# The assignment asks for coordinate-derived distance, so calculate it before deciding whether it is trustworthy.
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1.astype(float), lon1.astype(float), lat2.astype(float), lon2.astype(float)])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

completed['haversine_distance_km'] = haversine_km(completed['pickup_lat'], completed['pickup_lng'], completed['drop_lat'], completed['drop_lng'])
distance_corr = completed[['distance_km','haversine_distance_km']].corr().iloc[0,1]
within_01 = (completed['distance_km'].sub(completed['haversine_distance_km']).abs() <= 0.1).mean()
print(f'Haversine vs supplied distance correlation: {distance_corr:.4f}')
print(f'Rows within 0.1 km: {within_01:.2%}')
print('Decision: use supplied distance_km for prediction; retain Haversine only as a diagnostic.')

## Phase 4 — EDA
Analyze fare distribution, trip-distance distribution, fare vs distance, fare by hour/day/month, and correlations. Passenger-count plots are intentionally N/A.

In [ ]:
# Cell 6 — Core distributions and fare-vs-distance relationship
# Use a plotting sample only for speed; calculate statistics on the full completed dataset.
fig, axes = plt.subplots(1, 2, figsize=(13,4))
sns.histplot(completed['fare_amount'], bins=30, ax=axes[0]); axes[0].set_title('Fare Distribution')
sns.histplot(completed['distance_km'], bins=30, ax=axes[1]); axes[1].set_title('Trip-Distance Distribution')
plt.tight_layout(); plt.show()

sample = completed.sample(min(5000, len(completed)), random_state=42)
sns.scatterplot(data=sample, x='distance_km', y='fare_amount', alpha=.35, s=20)
plt.title('Fare vs Supplied Trip Distance'); plt.show()
print('Fare-distance correlation:', round(completed[['fare_amount','distance_km']].corr().iloc[0,1], 3))

In [ ]:
# Cell 7 — Time-pattern EDA
# These charts answer the business questions about fare by hour, weekday, and month.
hourly = completed.groupby('pickup_hour', as_index=False)['fare_amount'].mean()
day_avg = completed.groupby('day_name', as_index=False)['fare_amount'].mean()
month_avg = completed.groupby('pickup_month', as_index=False)['fare_amount'].mean()

fig, axes = plt.subplots(1, 3, figsize=(17,4))
sns.lineplot(data=hourly, x='pickup_hour', y='fare_amount', marker='o', ax=axes[0]); axes[0].set_title('Fare by Hour')
sns.barplot(data=day_avg, x='day_name', y='fare_amount', ax=axes[1]); axes[1].tick_params(axis='x', rotation=35); axes[1].set_title('Fare by Day')
sns.barplot(data=month_avg, x='pickup_month', y='fare_amount', ax=axes[2]); axes[2].set_title('Fare by Month')
plt.tight_layout(); plt.show()
display(hourly.sort_values('fare_amount', ascending=False).head())

In [ ]:
# Cell 8 — Correlation matrix
# Limit the matrix to supported numeric analysis variables from the actual dataset.
corr_cols = ['fare_amount','distance_km','pickup_hour','pickup_month','day_of_week']
sns.heatmap(completed[corr_cols].corr(), annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix'); plt.show()

## Phase 5–6 — Machine Learning and Evaluation
Authoritative training is performed in `train.py`: 80/20 split → 5-fold CV only on the training partition → select lowest mean CV RMSE → refit winner → evaluate once on untouched holdout.

In [ ]:
# Cell 9 — Load final CV comparison and untouched-test metrics
# Reading exported evidence keeps notebook reporting synchronized with the saved model artifact.
cv_results = pd.read_csv(CV_PATH)
final_metrics = pd.read_csv(FINAL_PATH)
metadata = json.loads(META_PATH.read_text(encoding='utf-8'))
display(cv_results.round(4))
display(final_metrics.round(4))
print('Selected model:', metadata['best_model'])
print('Selection method:', metadata['selection_method'])

In [ ]:
# Cell 10 — Recreate Actual vs Predicted on the untouched holdout
# Use the same fixed split and the same final serialized pipeline for consistency.
from sklearn.model_selection import train_test_split
features = metadata['features']
_, X_test, _, y_test = train_test_split(completed[features], completed['fare_amount'], test_size=.20, random_state=42, shuffle=True)
model = joblib.load(MODEL_PATH)
prediction = model.predict(X_test)
evidence = pd.DataFrame({'actual': y_test.to_numpy(), 'prediction': prediction})
sample_evidence = evidence.sample(min(5000, len(evidence)), random_state=42)
sns.scatterplot(data=sample_evidence, x='actual', y='prediction', alpha=.35, s=20)
low = min(sample_evidence.min()); high = max(sample_evidence.max())
plt.plot([low, high], [low, high], linestyle='--'); plt.title('Actual vs Predicted — Untouched Holdout'); plt.show()

In [ ]:
# Cell 11 — Inspect final model coefficients
# Coefficients show fitted influence after preprocessing; they are not causal effects.
preprocess = model.named_steps['preprocess']
estimator = model.named_steps['model']
effects = pd.DataFrame({'Feature': preprocess.get_feature_names_out(), 'Coefficient': np.ravel(estimator.coef_)})
effects['AbsoluteEffect'] = effects['Coefficient'].abs()
display(effects.sort_values('AbsoluteEffect', ascending=False).head(15))

## Phase 7 — Final Business Answers

1. **Strongest fare influence:** supplied trip distance is the strongest supported observed signal; coefficient charts are influence diagnostics, not causal proof.
2. **Distance effect:** strong positive relationship, Pearson correlation ≈ **0.871**.
3. **Passenger count significance:** **cannot be determined** because `passenger_count` is absent.
4. **Higher-fare hours:** **06:00** has the highest average fare in the full valid completed-trip data.
5. **Best model:** **Linear Regression**, selected by lowest mean training-only 5-fold CV RMSE.
6. **Final accuracy:** holdout MAE ≈ **2.474**, MSE ≈ **9.567**, RMSE ≈ **3.093**, R² ≈ **0.754**.
7. **New-trip estimate:** **Yes** — the Streamlit app loads the same saved pipeline and predicts from supported pre-trip inputs.